# CareerGuard Lite - Block 1 Baseline Comparison

> **Reproducibility note:** `careerguard_block1_baseline.py` is the canonical, locally verified runner and generates the versioned artifacts in `block1_results/` and `block1_models/`. This notebook remains an interactive Colab companion.

This Colab notebook compares three offline models on the EMSCAD / Kaggle `fake_job_postings.csv` dataset:

1. **Text-only baseline**
2. **Structured-only baseline**
3. **Combined text + structured model**

> Important: this dataset does not contain live browser URL/domain/link metadata. Therefore, the second model is named **structured-only**, not browser-metadata-only. Real browser extraction and browser latency must be tested later with the browser extension.

The notebook checks and cleans the dataset, removes exact duplicates before splitting, creates a stratified train/validation/test split, trains the three models fairly, selects thresholds on validation data, reports ranking and classification metrics, and exports results and predictions.


## 1. Install and import packages


In [ ]:
!pip -q install beautifulsoup4 joblib

In [ ]:
import os
import re
import json
import time
import random
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from google.colab import files

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    ndcg_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Ready.")


## 2. Upload the dataset

In [ ]:
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("No CSV file was uploaded.")

DATA_PATH = csv_files[0]
print("Using:", DATA_PATH)


## 3. Load and inspect the dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

required_columns = {
    "job_id", "title", "location", "department", "salary_range",
    "company_profile", "description", "requirements", "benefits",
    "telecommuting", "has_company_logo", "has_questions",
    "employment_type", "required_experience", "required_education",
    "industry", "function", "fraudulent"
}

missing_required = sorted(required_columns - set(df_raw.columns))
if missing_required:
    raise ValueError(f"Dataset is missing expected columns: {missing_required}")

print("\nTarget counts:")
print(df_raw["fraudulent"].value_counts(dropna=False))

print("\nTarget percentages:")
print((df_raw["fraudulent"].value_counts(normalize=True) * 100).round(2))


In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
}).sort_values("missing_percent", ascending=False)

display(missing_report)


## 4. Define feature groups

### Text-only baseline
Uses:
- title
- company profile
- description
- requirements
- benefits

### Structured-only baseline
Uses browser-observable or posting-structure fields available in EMSCAD:
- location
- department
- salary range
- employment type
- required experience
- required education
- industry
- function
- telecommuting
- company-logo indicator
- questions indicator

The structured fields are useful for the offline benchmark, but they are **not full live browser metadata**.

In [ ]:
TEXT_COLUMNS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
]

CATEGORICAL_COLUMNS = [
    "location",
    "department",
    "salary_range",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
]

BINARY_COLUMNS = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
]

TARGET_COLUMN = "fraudulent"
ID_COLUMN = "job_id"

print("Text columns:", TEXT_COLUMNS)
print("Categorical structured columns:", CATEGORICAL_COLUMNS)
print("Binary structured columns:", BINARY_COLUMNS)


## 5. Clean the data

The cleaning keeps missingness instead of deleting every incomplete row. Missing values can carry useful information.

Exact duplicate removal is done **before** splitting so that identical postings do not appear in both training and test sets.

In [ ]:
def clean_html_text(value) -> str:
    if pd.isna(value):
        return ""
    text = BeautifulSoup(str(value), "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw.copy()

# Validate and normalise target.
df = df[df[TARGET_COLUMN].isin([0, 1])].copy()
df[TARGET_COLUMN] = df[TARGET_COLUMN].astype(int)

# Clean text fields.
for col in TEXT_COLUMNS:
    df[col] = df[col].apply(clean_html_text)

# Make one labelled text field. Field labels help preserve section meaning.
df["combined_text"] = (
    "[TITLE] " + df["title"] +
    " [COMPANY] " + df["company_profile"] +
    " [DESCRIPTION] " + df["description"] +
    " [REQUIREMENTS] " + df["requirements"] +
    " [BENEFITS] " + df["benefits"]
)

# Normalise categorical fields.
for col in CATEGORICAL_COLUMNS:
    df[col] = df[col].fillna("MISSING").astype(str).str.strip()
    df.loc[df[col] == "", col] = "MISSING"

# Normalise binary fields.
for col in BINARY_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Remove rows with no usable text at all.
before_empty = len(df)
usable_text = df["combined_text"].str.replace(r"\[[A-Z]+\]", "", regex=True).str.strip()
df = df[usable_text.ne("")].copy()
removed_empty = before_empty - len(df)

# Detect exact feature duplicates before splitting.
# Conflicting duplicates have identical features but different labels; remove the whole conflict group.
feature_subset = ["combined_text"] + CATEGORICAL_COLUMNS + BINARY_COLUMNS
df["_feature_hash"] = pd.util.hash_pandas_object(df[feature_subset], index=False)

label_counts_by_hash = df.groupby("_feature_hash")[TARGET_COLUMN].nunique()
conflicting_hashes = set(label_counts_by_hash[label_counts_by_hash > 1].index)

before_conflicts = len(df)
df = df[~df["_feature_hash"].isin(conflicting_hashes)].copy()
removed_conflicting_duplicates = before_conflicts - len(df)

before_duplicates = len(df)
df = df.drop_duplicates(subset=feature_subset, keep="first").copy()
removed_duplicates = before_duplicates - len(df)

df = df.drop(columns=["_feature_hash"]).reset_index(drop=True)

print("Rows after cleaning:", len(df))
print("Removed empty rows:", removed_empty)
print("Removed rows from conflicting duplicate groups:", removed_conflicting_duplicates)
print("Removed same-label exact duplicates:", removed_duplicates)
print("\nClass counts after cleaning:")
print(df[TARGET_COLUMN].value_counts())


In [ ]:
# Inspect a few cleaned examples.
display(
    df[[ID_COLUMN, "title", "combined_text", TARGET_COLUMN]]
    .sample(3, random_state=RANDOM_STATE)
)


## 6. Create one fixed stratified split

The same train, validation and test rows are used for all three systems.

- 70% training
- 15% validation
- 15% testing

The validation set is used to choose a classification threshold. The test set is kept untouched until final evaluation.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

def split_summary(name, frame):
    counts = frame[TARGET_COLUMN].value_counts().to_dict()
    return {
        "split": name,
        "rows": len(frame),
        "real": counts.get(0, 0),
        "fake": counts.get(1, 0),
        "fake_percent": round(frame[TARGET_COLUMN].mean() * 100, 2),
    }

split_table = pd.DataFrame([
    split_summary("train", train_df),
    split_summary("validation", val_df),
    split_summary("test", test_df),
])

display(split_table)

assert set(train_df.index).isdisjoint(val_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(val_df.index).isdisjoint(test_df.index)


In [ ]:
# Save the exact split IDs for reproducibility.
split_ids = pd.concat([
    train_df[[ID_COLUMN]].assign(split="train"),
    val_df[[ID_COLUMN]].assign(split="validation"),
    test_df[[ID_COLUMN]].assign(split="test"),
], ignore_index=True)

split_ids.to_csv("block1_split_ids.csv", index=False)
print("Saved block1_split_ids.csv")


## 7. Build the three systems

All systems use logistic regression with balanced class weights. This keeps the first benchmark lightweight and fair.

No oversampling is applied to the validation or test sets.

In [ ]:
text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=40000,
            sublinear_tf=True,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
    ),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

structured_preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
    ("binary", binary_pipeline, BINARY_COLUMNS),
])

structured_model = Pipeline([
    ("features", structured_preprocessor),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
    ),
])

combined_preprocessor = ColumnTransformer([
    (
        "text",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            max_features=40000,
            sublinear_tf=True,
        ),
        "combined_text",
    ),
    ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
    ("binary", binary_pipeline, BINARY_COLUMNS),
])

combined_model = Pipeline([
    ("features", combined_preprocessor),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
    ),
])

models = {
    "Text-only baseline": {
        "model": text_model,
        "train_X": train_df["combined_text"],
        "val_X": val_df["combined_text"],
        "test_X": test_df["combined_text"],
    },
    "Structured-only baseline": {
        "model": structured_model,
        "train_X": train_df[CATEGORICAL_COLUMNS + BINARY_COLUMNS],
        "val_X": val_df[CATEGORICAL_COLUMNS + BINARY_COLUMNS],
        "test_X": test_df[CATEGORICAL_COLUMNS + BINARY_COLUMNS],
    },
    "Combined text + structured": {
        "model": combined_model,
        "train_X": train_df[["combined_text"] + CATEGORICAL_COLUMNS + BINARY_COLUMNS],
        "val_X": val_df[["combined_text"] + CATEGORICAL_COLUMNS + BINARY_COLUMNS],
        "test_X": test_df[["combined_text"] + CATEGORICAL_COLUMNS + BINARY_COLUMNS],
    },
}

y_train = train_df[TARGET_COLUMN].to_numpy()
y_val = val_df[TARGET_COLUMN].to_numpy()
y_test = test_df[TARGET_COLUMN].to_numpy()

print("Models prepared.")


## 8. Define fair evaluation functions

Ranking metrics use predicted fraud probabilities.

Classification metrics use a threshold chosen only from the validation set.

In [ ]:
def precision_at_k(y_true, scores, k=10):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]
    return float(y_true[top_indices].sum() / k)

def recall_at_k(y_true, scores, k=10):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    positives = y_true.sum()
    if positives == 0:
        return 0.0
    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]
    return float(y_true[top_indices].sum() / positives)

def ndcg_at_k(y_true, scores, k=10):
    y_true = np.asarray(y_true).reshape(1, -1)
    scores = np.asarray(scores).reshape(1, -1)
    return float(ndcg_score(y_true, scores, k=min(k, y_true.shape[1])))

def choose_f1_threshold(y_true, scores):
    thresholds = np.linspace(0.01, 0.99, 99)
    f1_values = [
        f1_score(y_true, (scores >= threshold).astype(int), zero_division=0)
        for threshold in thresholds
    ]
    best_index = int(np.argmax(f1_values))
    return float(thresholds[best_index]), float(f1_values[best_index])

def evaluate_scores(y_true, scores, threshold, k=10):
    predictions = (scores >= threshold).astype(int)

    return {
        f"Precision@{k}": precision_at_k(y_true, scores, k),
        f"Recall@{k}": recall_at_k(y_true, scores, k),
        f"NDCG@{k}": ndcg_at_k(y_true, scores, k),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "PR-AUC": average_precision_score(y_true, scores),
        "Threshold": threshold,
        "Predicted fake count": int(predictions.sum()),
    }


## 9. Train, validate and test all models

The timing below is **offline Colab timing**, not browser latency.

In [ ]:
results = []
prediction_frames = []
trained_models = {}

for model_name, bundle in models.items():
    print(f"\nTraining: {model_name}")

    model = bundle["model"]

    fit_start = time.perf_counter()
    model.fit(bundle["train_X"], y_train)
    fit_seconds = time.perf_counter() - fit_start

    val_scores = model.predict_proba(bundle["val_X"])[:, 1]
    threshold, validation_f1 = choose_f1_threshold(y_val, val_scores)

    test_start = time.perf_counter()
    test_scores = model.predict_proba(bundle["test_X"])[:, 1]
    test_seconds = time.perf_counter() - test_start

    metrics = evaluate_scores(y_test, test_scores, threshold, k=10)
    metrics.update({
        "Model": model_name,
        "Validation F1 at selected threshold": validation_f1,
        "Training time (s)": fit_seconds,
        "Offline test scoring time (s)": test_seconds,
        "Offline ms per test record": (test_seconds / len(test_df)) * 1000,
    })
    results.append(metrics)

    predictions = (test_scores >= threshold).astype(int)
    prediction_frames.append(pd.DataFrame({
        ID_COLUMN: test_df[ID_COLUMN].to_numpy(),
        "true_label": y_test,
        "model": model_name,
        "fraud_probability": test_scores,
        "selected_threshold": threshold,
        "predicted_label": predictions,
    }))

    trained_models[model_name] = model

results_df = pd.DataFrame(results).set_index("Model")
results_df = results_df[
    [
        "Precision@10", "Recall@10", "NDCG@10",
        "Precision", "Recall", "F1", "PR-AUC",
        "Threshold", "Predicted fake count",
        "Validation F1 at selected threshold",
        "Training time (s)", "Offline test scoring time (s)",
        "Offline ms per test record",
    ]
]

display(results_df.round(4))


## 10. Inspect confusion matrices and classification reports

In [ ]:
all_predictions = pd.concat(prediction_frames, ignore_index=True)

for model_name in models:
    subset = all_predictions[all_predictions["model"] == model_name]

    print("\n" + "=" * 80)
    print(model_name)
    print("Confusion matrix:")
    print(confusion_matrix(subset["true_label"], subset["predicted_label"]))

    print("\nClassification report:")
    print(
        classification_report(
            subset["true_label"],
            subset["predicted_label"],
            target_names=["Real", "Fake"],
            digits=4,
            zero_division=0,
        )
    )


## 11. Check more than one K value

The report asks for `@10`. Because the dataset is large and fake jobs are rare, it is also useful to inspect `@50` and `@100` without replacing the planned `@10` result.

In [ ]:
ranking_rows = []

for model_name in models:
    subset = all_predictions[all_predictions["model"] == model_name]
    y_true_model = subset["true_label"].to_numpy()
    scores_model = subset["fraud_probability"].to_numpy()

    for k in [10, 50, 100]:
        ranking_rows.append({
            "Model": model_name,
            "K": k,
            "Precision@K": precision_at_k(y_true_model, scores_model, k),
            "Recall@K": recall_at_k(y_true_model, scores_model, k),
            "NDCG@K": ndcg_at_k(y_true_model, scores_model, k),
        })

ranking_df = pd.DataFrame(ranking_rows)
display(ranking_df.round(4))


## 12. Plot a simple comparison

This chart is for presentation. Use the exported table as the main evidence.

In [ ]:
plot_metrics = results_df[["Precision@10", "Recall@10", "NDCG@10", "F1", "PR-AUC"]]

ax = plot_metrics.plot(kind="bar", figsize=(12, 6))
ax.set_title("Block 1 baseline comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 13. Export results and models

In [ ]:
os.makedirs("block1_results", exist_ok=True)
os.makedirs("block1_models", exist_ok=True)

results_df.reset_index().to_csv(
    "block1_results/block1_model_results.csv",
    index=False,
)

ranking_df.to_csv(
    "block1_results/block1_ranking_by_k.csv",
    index=False,
)

all_predictions.to_csv(
    "block1_results/block1_test_predictions.csv",
    index=False,
)

split_table.to_csv(
    "block1_results/block1_split_summary.csv",
    index=False,
)

missing_report.to_csv(
    "block1_results/block1_missing_report.csv",
)

split_ids.to_csv(
    "block1_results/block1_split_ids.csv",
    index=False,
)

for model_name, model in trained_models.items():
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("-", "_")
    )
    joblib.dump(model, f"block1_models/{safe_name}.joblib")

run_config = {
    "random_state": RANDOM_STATE,
    "dataset_file": str(DATA_PATH),
    "rows_after_cleaning": len(df),
    "removed_empty_rows": removed_empty,
    "removed_exact_duplicates": removed_duplicates,
    "removed_conflicting_duplicate_rows": removed_conflicting_duplicates,
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "text_columns": TEXT_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "binary_columns": BINARY_COLUMNS,
    "threshold_rule": "Maximise F1 on validation set separately for each model",
    "important_note": (
        "Structured-only EMSCAD fields are not full live browser metadata. "
        "Browser metadata and browser latency require a later extension experiment."
    ),
}

with open("block1_results/block1_run_config.json", "w", encoding="utf-8") as file:
    json.dump(run_config, file, indent=2)

print("Exports complete.")


In [ ]:
# Zip everything for download.
!zip -qr CareerGuard_Block1_Results.zip block1_results block1_models

files.download("CareerGuard_Block1_Results.zip")


## 14. How to interpret the result

The strongest Block 1 result would be:

- the combined model is at least comparable to the strongest baseline,
- it improves one or more ranking measures,
- it does not create an unacceptable false-positive pattern,
- the result is described as an **offline EMSCAD benchmark**, not proof of live browser usefulness.

Do not claim browser latency from this notebook. The recorded timing only measures offline scoring inside Colab.

A suitable preliminary conclusion template is:

> On the held-out EMSCAD test set, the combined text-and-structured model achieved [insert metrics], compared with [insert baseline results]. This supports the value of combining listing text and available structured posting signals under the evaluated offline conditions. Live browser metadata extraction, end-to-end warning latency, and decision-time usefulness remain separate browser-prototype evaluations.


## 15. Optional stronger final evaluation

For the final report, repeat the experiment with several random seeds or use stratified cross-validation. Report the mean and standard deviation. Keep the fixed test split above for the first prototype result and do not tune repeatedly on the test set.